In [ ]:
# Standard library imports
import os
import sys
import copy
import pickle
import warnings

# Third-party imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from rdkit import Chem

# Scikit-learn imports
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import LeaveOneOut, train_test_split

# Custom imports
from GCN import GCNModel, save_smiles_dicts, get_smiles_array

# Configuration
torch.manual_seed(8)
sys.setrecursionlimit(50000)
torch.backends.cudnn.benchmark = True
torch.set_default_dtype(torch.float32)
sns.set(color_codes=True)
%matplotlib inline

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


device(type='cuda')

In [ ]:
def process_smiles_data_and_setup_model(raw_filename, targets, batch_size=50, epochs=800, p_dropout=0.1, 
                                       fingerprint_dim=150, radius=4, T=3, weight_decay=2, learning_rate=3):
    """
    Process SMILES data and set up GCN model for molecular property prediction.
    
    Parameters:
    -----------
    raw_filename : str
        Path to the CSV file containing SMILES data
    targets : list
        List of target column names for prediction
    batch_size : int, default=50
        Batch size for training
    epochs : int, default=800
        Number of training epochs
    p_dropout : float, default=0.1
        Dropout probability
    fingerprint_dim : int, default=150
        Dimension of molecular fingerprints
    radius : int, default=4
        Radius for molecular graph construction
    T : int, default=3
        Number of GCN layers
    weight_decay : int, default=2
        Weight decay parameter (10^-weight_decay)
    learning_rate : int, default=3
        Learning rate parameter (10^-learning_rate)
    
    Returns:
    --------
    tuple
        Contains processed dataframe, feature dictionaries, model, optimizer, 
        loss functions, and other training parameters
    """
    # Setup filenames
    feature_filename = raw_filename.replace('.csv', '.pickle')
    prefix_filename = os.path.basename(raw_filename).replace('.csv', '')
    
    # Load data and binarize target columns
    df = pd.read_csv(raw_filename)
    for column in targets:
        df[column] = (df[column] > 0).astype(int)

    # Process SMILES strings
    valid_smiles = []
    canonical_smiles = []
    
    for smiles in df.SMILES.values:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol and len(mol.GetAtoms()) < 151:
                valid_smiles.append(smiles)
                canonical_smiles.append(Chem.MolToSmiles(mol, isomericSmiles=True))
        except Exception as e:
            warnings.warn(f"Failed to process SMILES {smiles}: {str(e)}")
    
    # Filter dataframe to keep only valid SMILES
    df = df[df["SMILES"].isin(valid_smiles)]
    df['cano_smiles'] = canonical_smiles
    
    # Load or create feature dictionaries
    feature_dicts = (pickle.load(open(feature_filename, "rb")) if os.path.isfile(feature_filename) 
                    else save_smiles_dicts(canonical_smiles, prefix_filename))
    
    # Filter to keep only SMILES with features
    remained_df = df[df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]
    
    if len(remained_df) < len(df):
        warnings.warn(f"There are {len(df) - len(remained_df)} uncovered SMILES in the dataset.", UserWarning)
    
    # Calculate class weights for each target
    weights = []
    for target in targets:
        neg_count = (remained_df[target] == 0).sum()
        pos_count = (remained_df[target] == 1).sum()
        total = neg_count + pos_count
        
        weights.append([total / neg_count, total / pos_count] if neg_count > 0 and pos_count > 0 else [1, 1])
    
    # Get feature dimensions from sample data
    sample_smiles = canonical_smiles[0]
    x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array([sample_smiles], feature_dicts)
    num_atom_features = x_atom.shape[-1]
    num_bond_features = x_bonds.shape[-1]
    
    # Setup model
    per_target_output_units_num = 2
    output_units_num = len(targets) * per_target_output_units_num
    
    model = GCNModel(T, radius, num_atom_features, num_bond_features,
                    fingerprint_dim, output_units_num, p_dropout)
    model.cuda()
    
    # Setup optimizer
    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)
    
    # Print model info
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    print(f"Number of parameters: {params}")
    
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"{name}: {param.data.shape}")
    
    # Setup loss functions
    loss_functions = [nn.CrossEntropyLoss(weight=torch.Tensor(weight).to(device), reduction='mean') 
                      for weight in weights]
    
    return (remained_df, feature_dicts, model, optimizer, loss_functions, 
            remained_df, canonical_smiles, batch_size, epochs)


def train(model, sample, optimizer, loss_function, targets, per_target_output_units_num, feature_dicts, device):
    """
    Train the model on a single sample.
    
    Parameters:
    -----------
    model : torch.nn.Module
        The GCN model to train
    sample : pd.Series
        Single training sample containing SMILES and target values
    optimizer : torch.optim.Optimizer
        Optimizer for model parameters
    loss_function : list
        List of loss functions for each target
    targets : list
        List of target column names
    per_target_output_units_num : int
        Number of output units per target (typically 2 for binary classification)
    feature_dicts : dict
        Dictionary containing molecular feature representations
    device : torch.device
        Device to run computations on
        
    Returns:
    --------
    float
        Training loss value
    """
    model.train()
    optimizer.zero_grad()
    
    x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array([sample.cano_smiles], feature_dicts)

    atoms_prediction, mol_prediction = model(torch.Tensor(x_atom).to(device), 
                                             torch.Tensor(x_bonds).to(device),
                                             torch.LongTensor(x_atom_index).to(device),
                                             torch.LongTensor(x_bond_index).to(device),
                                             torch.Tensor(x_mask).to(device))
    
    loss = 0.0
    for i, target in enumerate(targets):
        y_pred = mol_prediction[:, i * per_target_output_units_num:(i + 1) * per_target_output_units_num]
        y_val = sample[target]

        if y_val in [0, 1]:
            y_val = torch.LongTensor([y_val]).to(device)
            loss += loss_function[i](y_pred, y_val)

    loss.backward()
    optimizer.step()
    return loss.item()


def eval(model, sample, targets, per_target_output_units_num, feature_dicts, device, loss_function=None):
    """
    Evaluate the model on a single sample.
    
    Parameters:
    -----------
    model : torch.nn.Module
        The GCN model to evaluate
    sample : pd.Series
        Single sample containing SMILES and target values
    targets : list
        List of target column names
    per_target_output_units_num : int
        Number of output units per target (typically 2 for binary classification)
    feature_dicts : dict
        Dictionary containing molecular feature representations
    device : torch.device
        Device to run computations on
    loss_function : list, optional
        List of loss functions for each target (if provided, loss will be calculated)
        
    Returns:
    --------
    tuple
        (loss_value, predictions) where predictions is a list of probability scores
    """
    model.eval()
    with torch.no_grad():
        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array([sample.cano_smiles], feature_dicts)
        atoms_prediction, mol_prediction = model(torch.Tensor(x_atom).to(device), 
                                                 torch.Tensor(x_bonds).to(device),
                                                 torch.LongTensor(x_atom_index).to(device),
                                                 torch.LongTensor(x_bond_index).to(device),
                                                 torch.Tensor(x_mask).to(device))
        
        predictions = []
        loss = 0
        
        for i, target in enumerate(targets):
            y_pred = mol_prediction[:, i * per_target_output_units_num:(i + 1) * per_target_output_units_num]
            
            # Calculate loss if loss_function is provided
            if loss_function is not None:
                y_true = torch.tensor([sample[target]]).long().to(device)
                loss += loss_function[i](y_pred, y_true)
            
            # Get prediction
            pred = F.softmax(y_pred, dim=-1).cpu().numpy()[0, 1]
            predictions.append(pred)
        
    return loss.item() if loss_function is not None else 0, predictions


def train_test_loop(remained_df, model, optimizer, loss_functions, targets, feature_dicts, device,
                   per_target_output_units_num=2, num_epochs=800, patience=10, valid_size=0.2):
    """
    Perform leave-one-out cross-validation training and testing.
    
    Parameters:
    -----------
    remained_df : pd.DataFrame
        DataFrame containing molecular data with SMILES and targets
    model : torch.nn.Module
        The GCN model to train and evaluate
    optimizer : torch.optim.Optimizer
        Optimizer for model parameters
    loss_functions : list
        List of loss functions for each target
    targets : list
        List of target column names
    feature_dicts : dict
        Dictionary containing molecular feature representations
    device : torch.device
        Device to run computations on
    per_target_output_units_num : int, default=2
        Number of output units per target (typically 2 for binary classification)
    num_epochs : int, default=800
        Maximum number of training epochs per fold
    patience : int, default=10
        Early stopping patience (epochs without improvement)
    valid_size : float, default=0.2
        Fraction of training data to use for validation
        
    Returns:
    --------
    tuple
        (roc_auc_scores, classification_results) where roc_auc_scores is a list of 
        AUC scores for each target and classification_results is a dictionary of results
    """
    loo = LeaveOneOut()
    all_predictions = [[] for _ in targets]
    all_true_labels = [[] for _ in targets]
    all_Hosts = []
    classification = {}
    
    for fold_idx, (train_index, test_index) in enumerate(loo.split(remained_df)):
        print(f"Processing fold {fold_idx + 1}/{len(remained_df)}")
        train_valid_data = remained_df.iloc[train_index]
        test_sample = remained_df.iloc[test_index].squeeze()
        
        # Split train_valid_data into train and validation sets
        train_data, valid_data = train_test_split(train_valid_data, test_size=valid_size, random_state=42)
        
        # Reset model for each fold
        model.apply(lambda m: m.reset_parameters() if hasattr(m, 'reset_parameters') else None)
        
        # Initialize best model and early stopping variables
        best_model = None
        best_val_loss = float('inf')
        patience_counter = 0
        
        # Training loop
        for epoch in range(num_epochs):
            # Train phase
            model.train()
            train_loss = 0
            for _, sample in train_data.iterrows():
                batch_loss = train(model, sample, optimizer, loss_functions, targets, 
                                  per_target_output_units_num, feature_dicts, device)
                train_loss += batch_loss
            
            avg_train_loss = train_loss / len(train_data)
            
            # Validation phase
            model.eval()
            valid_loss = 0
            with torch.no_grad():
                for _, sample in valid_data.iterrows():
                    loss, _ = eval(model, sample, targets, per_target_output_units_num, 
                                  feature_dicts, device, loss_functions)
                    valid_loss += loss
            
            avg_valid_loss = valid_loss / len(valid_data)
            
            # Print progress every 50 epochs or on early stopping
            if (epoch + 1) % 50 == 0 or epoch == 0:
                print(f"  Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Valid Loss: {avg_valid_loss:.4f}")
            
            # Save the best model based on validation loss
            if avg_valid_loss < best_val_loss:
                best_val_loss = avg_valid_loss
                best_model = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Early stopping check
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
        
        # Load the best model for evaluation
        model.load_state_dict(best_model)
        
        # Evaluation on test sample
        _, predictions = eval(model, test_sample, targets, per_target_output_units_num, 
                             feature_dicts, device)
        
        # Invert predictions
        inverted_predictions = [1 - pred for pred in predictions]
        
        # Get the Host (SMILES) for this test point
        host = test_sample.Host
        all_Hosts.append(host)
        
        # Create a dictionary for this test sample
        classification[host] = {}
        
        for i, target in enumerate(targets):
            raw_output = inverted_predictions[i]
            threshold = 0.5
            true_label = test_sample[target]
            
            # Add to the classification dictionary
            classification[host][target] = (raw_output, threshold, true_label)
            
            # Collect predictions and true labels
            all_predictions[i].append(raw_output)
            all_true_labels[i].append(true_label)
            
            # Print concise results
            pred_label = 1 if raw_output > threshold else 0
            print(f"  {target}: Pred={pred_label} (p={raw_output:.3f}), True={true_label}")
        
        print()

    # Calculate ROC AUC scores
    roc_auc_scores = [roc_auc_score(all_true_labels[i], all_predictions[i]) for i in range(len(targets))]
    
    return roc_auc_scores, classification

                        